## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/Q8HEZn0.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

---

# Breakout Room #1
## Deep Research Foundations

In this breakout room, we'll understand the architecture and components of the Open Deep Research system.

## Task 1: Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

In [23]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

## Task 2: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

## Task 3: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

## Task 4: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 5: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Question #1:

Explain the interrelationships between the three states (Agent, Supervisor, Researcher). Why don't we just make a single huge state?

##### Answer:
**Agent State**: connects input question with the steps used to find an answer, high-level instructions for converting plain-language user input into a command for the supervisor layer to fulfill<br><br>
**Supervisor State**: gives overview of how high-level tasks inferred by user's question are delegated to lower, more specialized sub-agents, also includes data used to convert low-level agent respones back into a broader response that actually answers the user's initial question<br><br>
**Researcher State**: tracks low-level processes performed by individual research agents to get answers to specific sub-questions they are tasked with finding, tracks tool calls used and count of tool use iterations<br><br>
Keeping all of these state components in a single huge state would make it difficult for the high-level deep research agent to keep track of which sub-tasks have been completed versus still in progress or failed. Breaking out the various state components makes it faster for the low-level research agents to access the state data that they care about, without bloating their own context with extraneous or unrelated facts that are only relevant for other parts of the RAG system.

## Question #2:

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

##### Answer:
**Advantages**:
- hide complex and agent-specific components in another file so that its easier to read and interpret the high-level flow of the RAG system without delving into specific implemenation details
- can break out files that are only related to one specific agent and not needed by the others
- can logically group functions and class definitions based on which knowledge subset they operate on<br><br>

**Disadvantages**:
- you have to know which file contains agent-specific class or state definitions to lookup implementation details
- some tools may exist in overlapping namespaces, making it difficult to know which version of a specific tool is being utilized at any given point in the RAG process
- harder to determine exactly what a full RAG system does without checking in each supporting file for implementation-specific logic and arguments / return values

## Activity #1: Explore the Prompts

Open `open_deep_library/prompts.py` and examine one of the prompt templates in detail.

**Requirements:**
1. Choose one prompt template (clarify, brief, supervisor, researcher, compression, or final report)
2. Explain what the prompt is designed to accomplish
3. Identify 2-3 key techniques used in the prompt (e.g., structured output, role definition, examples)
4. Suggest one improvement you might make to the prompt

**YOUR CODE HERE** - Write your analysis in a markdown cell below

_summarize_webpage_prompt_<br><br>
**prompt purpose**: distill the contents of a webpage into a summary that contains key details, quotes, lists, chronolgoical event orders and dates from the page<br><br>
**techniques used**: structured output (summary / key_excerpts attributes in example call / responses), examples of input text and expected output summaries<br><br>
**potential improvement**: show specific locations in the input document from which key_excerpts are derived, show an example of how the output of the prompt will be used by downstream consuming agents

---

# Breakout Room #2
## Building & Running the Researcher

In this breakout room, we'll explore the node functions, build the graph, and run investment research.

## Task 6: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit x 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 7: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START -> researcher -> researcher_tools -> compress_research -> END
               ^            |
               +------------+ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START -> supervisor -> supervisor_tools -> END
            ^              |
            +--------------+ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START -> clarify_with_user -> write_research_brief -> research_supervisor -> final_report_generation -> END
                 |
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher -> researcher_tools -> compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor -> supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 8: Running the Deep Researcher

Now let's see the system in action! We'll use it to research investment strategies for portfolio diversification into alternatives.

### Setup

We need to:
1. Set up the investment research request
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("\u2713 Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (1 concurrent researcher for cost control)
- **Clarification enabled** (will ask if research scope is unclear)

In [21]:
# Configure for OpenAI with moderate settings
config = {
    "configurable": {
        # Model configuration - using GPT-4.1 for everything
        "research_model": "openai:gpt-4.1",
        "research_model_max_tokens": 10000,
        
        "compression_model": "openai:gpt-4.1",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "openai:gpt-4.1",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "openai:gpt-4.1-mini",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("\u2713 Configuration ready")
print(f"  - Research Model: GPT-4.1")
print(f"  - Max Concurrent Researchers: 1")
print(f"  - Max Iterations: 2")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: GPT-4.1
  - Max Concurrent Researchers: 1
  - Max Iterations: 2
  - Search API: Tavily


### Execute the Investment Research

Now let's run the research! We'll ask the system to research evidence-based strategies for incorporating alternative investments into a traditional portfolio.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [17]:
# Create our investment research request
research_request = """
I want to diversify my portfolio into alternative investments. I currently have:
- 60% equities, 30% bonds, 10% cash
- No exposure to alternatives like reinsurance, private equity, or real estate
- Limited understanding of how alternatives can reduce portfolio risk

Please research the best evidence-based strategies for incorporating alternative investments into a traditional portfolio and create a comprehensive diversification plan.
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

To create the most relevant diversification plan for your portfolio, I need to understand a few key details:

**Investment Parameters:**
- What is your approximate portfolio size or investment amount?
- What is your investment timeline/horizon (e.g., 5, 10, 20+ years)?
- What is your risk tolerance level (conservative, moderate, aggressive)?

**Investment Preferences:**
- Do you have any preferences or restrictions on specific alternative investment types?
- Are you open to both liquid and illiquid alternative investments?
- Do you prefer direct investments or fund-based approaches (REITs, private equity funds, etc.)?

**Geographic and Regulatory Context:**
- What is your country/region of residence for tax and regulatory considerations?
- Are there any specific tax considerations or investment account types (401k, IRA, taxable, etc.) I should factor in?

This information will help me tailor the research to provide evidence-based

## Task 9: Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided specific details about your portfolio, it likely proceeded without asking clarifying questions.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` to delegate to researchers
- Each delegation specified a focused research topic (e.g., reinsurance strategies, private equity allocation, real estate investment trusts)

### Phase 4: Parallel Research
Researchers worked on their assigned topics:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive portfolio diversification plan with:
- Well-structured sections
- Evidence-based recommendations
- Practical action items
- Sources for further reading

## Task 10: Key Takeaways & Next Steps

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

## Question #3:

What are the trade-offs of using parallel researchers vs. sequential research? When might you choose one approach over the other?

##### Answer:
Parallel researchers allow you to extract more information from disparate sources in a shorter amount of time, especially if the info theyre retrieving isn't related. Sequential researchers allow you to augment info that has already been retrieved with more helpful info derived from other agents, giving a much more thurough response than you would have otherwise gotten with one agent. The sequential approach works for complex questions that require info from various sources to be summarized and combined to produce the final answer. If you expect to have conditional branching logic in the RAG answering process than sequential agents may be better suited to taking paths conditional on the findings of agents before them. The parallel approach works better if you have many simpler questions that need to be answered from a wide array of input sources that may not have a lot of shared context.

## Question #4:

How would you adapt this deep research architecture for a production investment research application? What additional components would you need?

##### Answer:
This deep research architecture could be adapted with a more durable memory store, like an external database, for high-volume information sources. It would also benefit from an addittional node that evaluates every answer the RAG system gives against some quantitative framework of accuracy, cost-efficiency and output reproducibility. 

## Activity #2: Custom Investment Research

Using what you've learned, run a custom investment research task.

**Requirements:**
1. Create an investment-related research question (reinsurance, private equity, real estate, commodities)
2. Modify the configuration for your use case
3. Run the research and analyze the output
4. Document what worked well and what could be improved

**Experiment ideas:**
- Research reinsurance strategies for different risk profiles
- Compare different alternative investment vehicles
- Investigate private equity fund structures and fee models
- Explore commodities as inflation hedges

**YOUR CODE HERE**

In [24]:
# YOUR CODE HERE
# Create your own investment research request and run it

my_investment_request = """
I'm an accredited investor evaluating reinsurance and insurance-linked securities (ILS)
as a portfolio diversifier. I want to understand:

1. How catastrophe bonds (cat bonds) have performed over the last 5 years compared to
   traditional fixed income (investment-grade corporates and Treasuries).
2. What are the key risks of ILS investing — including climate change, model risk,
   and liquidity risk — and how are they priced?
3. What allocation percentage to ILS/reinsurance do institutional investors (pensions,
   endowments) typically hold, and what evidence supports those levels?
4. Are there accessible vehicles (funds, ETFs, interval funds) for accredited but
   non-institutional investors to gain ILS exposure?

Please provide a data-driven research report with specific return figures, risk metrics,
and actionable recommendations for a moderate-risk investor considering a 5-10% ILS allocation.
"""

# Optionally modify the config
my_config = {
    "configurable": {
        "research_model": "openai:gpt-4.1",
        "research_model_max_tokens": 10000,
        "compression_model": "openai:gpt-4.1",
        "compression_model_max_tokens": 8192,
        "final_report_model": "openai:gpt-4.1",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "openai:gpt-4.1-mini",
        "summarization_model_max_tokens": 8192,
        "allow_clarification": False,       # Skip clarification — the prompt is detailed
        "max_concurrent_research_units": 2,  # 2 parallel researchers for the 4 sub-questions
        "max_researcher_iterations": 3,      # Allow 3 delegation rounds for thorough coverage
        "max_react_tool_calls": 4,           # 4 tool calls per researcher for deeper dives
        "search_api": "tavily",
        "max_content_length": 50000,
        "thread_id": str(uuid.uuid4()),
    }
}

# Run the custom research
async def run_custom_research(request, cfg):
    """Run a custom research workflow and display results."""
    print("Starting custom investment research...\n")

    async for event in graph.astream(
        {"messages": [{"role": "user", "content": request}]},
        cfg,
        stream_mode="updates",
    ):
        for node_name, node_output in event.items():
            if node_output is None:
                continue
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")

            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")

            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")

            elif node_name == "supervisor":
                print("\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")

            elif node_name == "supervisor_tools":
                print("\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")

            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n{'='*60}")
                    print("FINAL REPORT GENERATED")
                    print(f"{'='*60}\n")
                    display(Markdown(node_output["final_report"]))

    print(f"\n{'='*60}")
    print("Custom research workflow completed!")
    print(f"{'='*60}")

await run_custom_research(my_investment_request, my_config)

Starting custom investment research...



/Users/alexei.naumann/Desktop/AIE/AI-Engineering/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ResearchQuestion(research...at them as open-ended.'), input_type=ResearchQuestion])
  return self.__pydantic_serializer__.to_python(



Node: write_research_brief

Research Brief Generated:
I want a comprehensive, data-driven analysis to evaluate the role of catastrophe bonds (cat bonds) and broader insurance-linked securities (ILS) as portfolio diversifiers for a moderate-risk accredited investor. Specifically, I want to understand: (1) How have cat bonds performed, in terms of returns and risk metrics (e.g., volatility, drawdowns, correlation with traditional asset classes), compared to traditional fixed income assets (investment-grade corporate bonds and US Treasuries) over the ...


/Users/alexei.naumann/Desktop/AIE/AI-Engineering/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Summary(summary='The Bloo..., JPY, KRW currencies.'), input_type=Summary])
  return self.__pydantic_serializer__.to_python(
/Users/alexei.naumann/Desktop/AIE/AI-Engineering/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Summary(summary="Lane Fin...he mid-single digits."'), input_type=Summary])
  return self.__pydantic_serializer__.to_python(
/Users/alexei.naumann/Desktop/AIE/AI-Engineering/08_Advanced_Retrieval_and_Deep_Research/.venv/lib/python3.13/si


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Catastrophe Bonds and Insurance-Linked Securities (ILS) for Moderate-Risk Accredited Investors: Performance, Risks, Allocations, and Access

## Executive Summary

This report provides a comprehensive, data-driven analysis of catastrophe bonds (cat bonds) and broader insurance-linked securities (ILS) as portfolio diversifiers for moderate-risk accredited investors. It examines recent (2021–2025) performance, risk metrics, key risks and how they are priced, institutional allocation practices, and available investment vehicles for non-institutional accredited investors. Actionable recommendations are provided for portfolio allocations and due diligence.

---

## 1. Catastrophe Bonds vs. Traditional Fixed Income (2021–2025): Returns and Risk Metrics

### 1.1 Performance Overview

**Cat Bonds (Swiss Re Global Cat Bond Index):**
- **2023**: 19.69%  
- **2024**: 17.3%  
- **2025**: 11.40%  
- **Cumulative (2021–2025)**: 61%  
- **Average Annualized**: ~12.5%  
- These represent the best multi-year performance since index inception. Cat bonds delivered double-digit returns for three consecutive years for the first time, reflecting strong market conditions and high demand[1][2][3][4][5].

**Investment-Grade US Corporate Bonds (Bloomberg US Corporate Bond Index):**
- **2025**: 7.56%  
- **US bond market broader index (Bloomberg US Aggregate)**: 7.3% in 2025  
- 2021–2025 saw mid-high single-digit returns overall, with peak performance during declining rate years and underperformance (even negative returns) during periods of aggressive rate hikes[6][7][8][12][24].

**US Treasuries (Bloomberg US Treasury Bond Index):**
- **2025**: 6.17%  
- **Yield to maturity (Nov 2025)**: 3.83%  
- **Average duration**: 5.95–7.71 years  
- Treasuries remained a safe-haven throughout market turbulence and rate volatility, but offered the lowest returns of the three categories in the recent years due to a lower coupon base[13][14][15][24].

### 1.2 Risk Metrics and Drawdowns

- **Volatility**: Cat bonds posted volatility and drawdowns meaningfully below equities and comparable to or below high-yield bonds. Most years are positive, with rare negative events linked to catastrophic insurance losses (e.g., major hurricanes). Historical Sharpe ratios for cat bonds in 2025 were around 1.3, compared to sub-1 for most bond indices[17][19][23].
- **Drawdowns**: Corporate bonds and Treasuries experienced pronounced negative years amid rate shocks (notably in 2022), with drawdowns as high as -13% for core bond indices in that period; cat bond indices avoided such sharp drops[6][20].
- **Correlations**: Cat bonds exhibit very low (near-zero) correlation to both equities and traditional fixed income. Their performance is primarily driven by insurance event risks rather than general market or credit risks, enabling significant portfolio diversification benefits[16][19][23].

### 1.3 Key Observations for Portfolio Diversification

- Cat bonds have consistently outperformed comparable fixed income segments over the last five years, with lower volatility and drawdown risk, and with strong risk-adjusted returns (Sharpe ratios exceeding 1).
- Diversification benefits are robust: cat bonds can reduce a portfolio's sensitivity to macroeconomic shocks and interest rate cycles due to their unique risk drivers.  
- Cat bonds increasingly form a core "structural" allocation in institutional and sophisticated private portfolios[17][23].

---

## 2. Key Risks of ILS Investing: Climate, Model, and Liquidity

### 2.1 Main Risk Factors

**Climate Change Risk**
- More volatile/extreme weather and climate-driven perils (e.g., hurricanes, wildfires) can increase default probability in a cat bond or ILS portfolio. Recent climate shifts led to broader risk spreads in 2023–2024 and greater premium for perils with rising modeled losses[4][18].
- Despite this, increased modeling sophistication and event-specific underwriting have contained loss ratios: the historical cumulative loss rate since market inception is ~2.7%[18].

**Model (Event/Trigger) Risk**
- Cat bonds depend on probabilistic catastrophe models. Model error or "basis risk" (divergence of model from reality or insured loss from bond payout) could create mismatches for investors.
- Innovation in triggers (parametric, indemnity, industry loss) diversifies risk but also introduces new sources of uncertainty. Due diligence on trigger mechanics, modeling methodology, and diversification across perils reduces exposure[18][23].

**Liquidity Risk**
- Cat bonds are generally more liquid than most private fund ILS, with daily to weekly secondary market liquidity. Liquidity can diminish sharply after major events or during active natural catastrophe seasons, potentially leading to wider bid-ask spreads and valuation uncertainty[2][4][20][23].
- Private ILS (such as collateralized reinsurance) can entail much longer lock-up periods, with interval or gated redemptions depending on the structure.

### 2.2 Pricing and Impact on Investor Outcomes

- **Spread Risk/Premium**: Cat bond yields/spreads reflect expected event loss, risk premium for insured perils (which rises post-major events), and liquidity premium. Spreads have recently compressed (from over 11% in early 2023 to just over 5% by 2026) as capital floods in post-strong returns, but net yields remain above high-yield fixed income[2][16].
- **Event Losses**: Most years are positive, but large payout years after major hurricanes or earthquakes can cause episodic drawdowns. The market's record demonstrates rare but manageable negative periods, buttressed by recovery in following years through repricing[18][19][23].
- **Liquidity and Valuation**: Interval and private vehicles can be subject to gates, requiring careful assessment of manager communication, fund structure, and underlying asset mix[23].

### 2.3 Risk Mitigation in Practice

- Diversification across peril types and geographies, rigorous model selection, and allocation through experienced managers can meaningfully reduce idiosyncratic risk.
- Institutional practice typically includes portfolio stress-testing and scenario modeling for climate and event concentration risk[4][23].

---

## 3. Institutional Allocation to ILS/Reinsurance

### 3.1 Typical Asset Allocation

- **Allocation Range**: Institutional investors such as pensions, endowments, and foundations typically allocate **2%–5%** of portfolio assets to ILS or cat bonds, with some sophisticated pools running allocations as high as **5%–10%**[19][23].
- **Survey Evidence**: The Investor Survey by Artemis (2025) and public case studies (e.g., Ontario Teachers’, Swiss Pensions, major US Endowments) confirm this range. The most common allocation is 2–4% for most US and European public pension plans, often within the alternative or "real assets" sleeve[23].
- **Rationale**: These allocations are pursued for uncorrelated return/risk, attractive risk-adjusted yield, and portfolio drawdown mitigation, as observed in periods where core bonds or equities sold off but ILS returns remained stable[19][23].

### 3.2 Supporting Institutional Case Studies

- **Ontario Teachers’**: Steadily increased ILS allocation since 2021, now exceeding 5%, citing consistent positive returns and low drawdowns; uses a combination of direct cat bond holdings and manager mandates.
- **US Ivy League Endowments** (aggregate): Position ILS as a diversifier within the alternatives portfolio. Allocations range from 2%–4%, with emphasis on fund managers specializing in diversified cat risk[23].

---

## 4. Access for Accredited but Non-Institutional Investors

### 4.1 Accessible Vehicles

**Funds and Interval Funds**
- Several UCITS ILS/cat bond funds are available in Europe and select global markets, with minimums from $10,000 to $100,000.  
- US investors have access through 1940 Act interval funds, which offer quarterly liquidity and focus on cat bond exposure (e.g., Stone Ridge Reinsurance Risk Premium Interval Fund, Hoya Capital High Yield ILS Fund). Minimums typically start at $2,500–$25,000[23].

**ETFs**
- ILS-focused ETFs (e.g., AXS ILS Cat Bond ETF) available in the US. These typically provide fully transparent, daily-liquidity portfolios of cat bonds and related instruments, with low minimums and expense ratios ranging from 0.39%–0.75%[23].

**Private Funds**
- High minimum investment ($100,000+) and more frequent use of lockups/gates, sometimes only available to "qualified purchasers" rather than just "accredited investors".

### 4.2 Key Characteristics

- **Fees:**  
  - UCITS funds: 0.70%–1.50% annual, no or low-load.
  - US Interval Funds: 1.0%–2.1% total expense ratio.
  - ETFs: 0.39%–0.75% annual.
- **Liquidity:**  
  - ETFs: Daily.
  - Interval funds: Quarterly (with some gating risk).
  - Private funds: Annual+ (subject to loss adjustment period).
- **Track Records:**  
  - Leading vehicles demonstrate 5–15 year significantly positive return history, with only rare negative years (event specific). Fund performance generally tracks the Swiss Re Cat Bond Index closely, net of fees[23].
- **Access Limitations:**  
  - Most vehicles are "accredited investor only". Interval funds and ETFs are the most accessible for US residents; UCITS are more available for offshore and EU-based investors.

### 4.3 Regional and Provider Variations

- **US:** Focus on 1940 Act interval funds and ETFs.
- **Europe/Asia:** UCITS cat bond funds from major managers (e.g., GAM, Swiss Re, Schroders).
- **Offshore:** Cayman/Bermuda private funds for very large allocations.

---

## 5. Actionable Recommendations for Moderate-Risk Investors (5–10% ILS Allocation)

- Consider allocating 5–10% of a multi-asset portfolio to ILS/cat bonds for robust diversification, yield enhancement, and drawdown protection, especially in the current low-to-moderate yield environment for traditional fixed income[19][23].
- For moderate liquidity preference, favor cat bond-focused ETFs or US interval funds, which offer transparency, portfolio diversification, and regulatory oversight. For higher. yield and complexity tolerance, private ILS funds or bespoke mandates may be considered (with professional advice).
- Prioritize funds/managers with:
    - Deep portfolio diversification by peril and region
    - Strong, multi-year track record vs. index (Swiss Re Cat Bond Index)
    - Clear communication on risk, liquidity, and valuation
    - Reasonable fees (preferably <1.5% annual where possible)
- Limit allocation to perils or regions with elevated climate/model uncertainty unless fully understood and accepted within portfolio risk budgets.
- Regularly monitor allocation size as ILS markets are cyclical. Post-run-up periods (e.g., after three consecutive years of double-digit returns) tend to see spread compression and possible lower prospective returns.
- Employ ongoing portfolio monitoring for liquidity, risk concentration, and model changes.

---

## 6. Conclusion

Catastrophe bonds and related ILS provide moderate-risk accredited investors a compelling, data-supported diversification tool. They have outperformed traditional fixed income in recent years, with lower sensitivity to macroeconomic conditions and rare, manageable drawdowns. Risks are unique but well-understood and can be mitigated by prudent manager selection, portfolio construction, and careful vehicle choice. The 5–10% allocation used by many sophisticated investors is supported by robust evidence, with multiple accessible funds and ETFs now available to non-institutional accredited investors. Ongoing diligence on climate, model, and liquidity developments remains essential for successful outcomes.

---

### Sources

[1] Swiss Re Global Cat Bond Performance Index returns 11.40% for 2025: https://www.artemis.bm/news/swiss-re-global-cat-bond-performance-index-returns-11-40-for-2025/  
[2] Catastrophe-Bond Risk Premia Drop to Rates Last Seen in 2022: https://www.insurancejournal.com/news/international/2026/02/17/858179.htm  
[3] Swiss Re Global Cat Bond Performance Index returns 11.40% for 2025 (ILS Course): https://ils-course.com/2026/01/12/swiss-re-global-cat-bond-performance-index-returns-11-40-for-2025/  
[4] Insurance-Linked Securities Market Insights - Swiss Re (February 2025): https://www.swissre.com/dam/jcr:e4989a50-1a40-49fd-bba2-87a4b6da1015/ils-market-insights-february-2025.pdf  
[5] Insurance-Linked Securities Market Insights - Swiss Re (February 2024): https://www.swissre.com/dam/jcr:bb189e59-a15f-49df-a250-07b2c6b2d9bd/2024-02-sr-ILS-market-insights-feb-2024.pdf  
[6] Bloomberg US Corporate - Live Performance & Historical Returns: https://ycharts.com/indices/%5EBBUSCOTR  
[7] Bloomberg Fixed Income Indices: https://www.bloomberg.com/professional/products/indices/fixed-income/  
[8] Bloomberg US Corporate Total Return Value Unhedged USD | Indices: https://www.bloomberg.com/professional/products/indices/quote/LUACTRUU:IND  
[12] Looking back at 2025: Fixed income | Insights - Bloomberg.com: https://www.bloomberg.com/professional/insights/markets/looking-back-at-2025-fixed-income/  
[13] Bloomberg US Treasury - Live Performance & Historical Returns: https://ycharts.com/indices/%5EBBUSTTR  
[14] Bond Volatility Is Collapsing as US Shutdown Creates Data Void: https://www.bloomberg.com/news/articles/2025-10-06/bond-volatility-is-collapsing-as-us-shutdown-creates-data-void  
[15] SPDR® Bloomberg US Treasury Bond UCITS ETF (Dist) Factsheet: https://www.ssga.com/library-content/products/factsheets/etfs/emea/factsheet-emea-en_gb-sybt-gy.pdf  
[16] Cat bonds deliver in 2025. Demonstrate low correlation, spreads exceed high yield - Swiss Re: https://www.artemis.bm/news/cat-bonds-deliver-in-2025-demonstrate-low-correlation-spreads-exceed-high-yield-swiss-re/  
[17] A combination of attributes that are increasingly rare in fixed income - Swiss Re cat bonds: https://www.gam.com/en/our-thinking/outlook-2026/swiss-re-cat-bonds  
[18] The 2025 Hurricane Season and its Potential Impact on Catastrophe Bond Investors: https://ilsetf.com/news-insights/the-2025-hurricane-season-and-its-potential-impact-on-catastrophe-bond-investors  
[19] The Case for Cat Bonds & ILS: Elevating Diversification and Returns - Sage Advisory: https://www.sageadvisory.com/article/the-case-for-cat-bonds-ils-elevating-diversification-and-returns-in-2026  
[20] Fixed Income: Weathering the storm - Victory Capital: https://investor.vcm.com/insights/market-insights/fixed-income-weathering-the-storm  
[23] Cat Bonds: What Allocators Need to Know Now - Resonanz Capital: https://resonanzcapital.com/insights/cat-bonds-what-allocators-need-to-know-now  
[24] Bond Market Wraps Up 2025 With Broad Gains - Morningstar: https://www.morningstar.com/bonds/bond-market-wraps-up-2025-with-broad-gains


Custom research workflow completed!
